In [2]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import sklearn
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm

from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset
import json

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.download_leaderboard import MMLU_SUBSCENARIOS
# from utils_for_notebooks import read_per_model_info
from utils_for_notebooks import pad_predictions
from scripts.evaluate_mmlu import evaluate_mmlu
from scripts.extract_model_outputs_from_raw_data_v2 import MMLU_PRO_SCENARIO_SUFFIX
sys.path.pop(0);


CACHE_DIR = "./cache_dir"


## functions

In [9]:
def _get_scenario_key(entry):
    """Return the key that ends with __leaderboard_mmlu_pro, or None."""
    if not isinstance(entry, dict):
        return None
    for k in entry:
        if k.endswith(MMLU_PRO_SCENARIO_SUFFIX):
            return k
    return None


def extract_source_outputs_from_v2_raw(raw_path, pad_to_size=31):
    """
    Load v2 leaderboard raw pickle and build source_outputs dict.

    Raw pickle format: {model_name: {scenario_name: {correctness, resps, ...}}}

    pad_to_size: size of the last dimension of predictions (pad with -inf).
    """
    raw = load_pickle(raw_path)

    # models = []
    scenario_key = None
    n_questions = None

    models_dict = {}

    for model_name, scenarios_dict in tqdm(raw.items()):
        if not isinstance(scenarios_dict, dict):
            continue
        sk = _get_scenario_key(scenarios_dict)
        if sk is None:
            continue
        rec = scenarios_dict[sk]
        if rec is None:
            continue
        correctness = rec.get("correctness")
        if correctness is None:
            continue
        try:
            nc = len(correctness)
        except TypeError:
            continue
        if scenario_key is None:
            scenario_key = sk
            n_questions = nc
        # elif sk != scenario_key or nc != n_questions:
        elif nc != n_questions:
            continue
        # models.append((model_name, rec))
        models_dict[model_name] = rec

    # if not models or scenario_key is None or n_questions is None:
    #     raise ValueError(
    #         "No valid model data found in raw pickle; need at least one model "
    #         "with non-None correctness for the MMLU-Pro scenario."
    #     )

    # n_models = len(models)
    # n_answers = 1

    # correctness_arr = np.zeros((n_models, n_questions, 1), dtype=np.float64)
    # predictions_arr = np.full(
    #     (n_models, n_questions, pad_to_size), -np.inf, dtype=np.float64
    # )

    return models_dict

    # for i, (model_name, rec) in enumerate(models):
    #     corr = rec["correctness"]
    #     if hasattr(corr, "__iter__") and not isinstance(corr, (str, bytes)):
    #         corr = list(corr)
    #     else:
    #         corr = [float(corr)] * n_questions
    #     correctness_arr[i, :, 0] = np.array(
    #         corr[:n_questions], dtype=np.float64
    #     )

    #     resps = rec.get("resps") or rec.get("filtered_resps")
    #     pred = _resps_to_prediction_indices(resps, n_questions, max_answers=1)
    #     predictions_arr[i, :, :1] = pred[:, :1]

    # # Same format as two_stages.build_outputs_dict / source_outputs.pkl
    # scenarios_map = {scenario_key: list(range(n_questions))}
    # models_map = {name: i for i, (name, _) in enumerate(models)}
    # datapoints_map = {i: i for i in range(n_questions)}

    # return {
    #     "predictions": predictions_arr,
    #     "correctness": correctness_arr,
    #     "Models": models_map,
    #     "Datapoints": datapoints_map,
    #     "Scenarios": scenarios_map,
    # }

## check remote responses

### Save single model

In [ ]:
raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
# raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
models_dict = extract_source_outputs_from_v2_raw(raw_path)
# 12 min loading

100%|██████████| 6505/6505 [00:00<00:00, 755557.79it/s]


In [30]:
for key in models_dict.keys():
    if "JungZoona" in key:
        print(key)
        print(models_dict[key].keys())
        break


open-llm-leaderboard/JungZoona__T3Q-qwen2.5-14b-v1.0-e3-details
dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])


In [31]:
import pickle

# model_name = 'open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'
model_name = 'open-llm-leaderboard/JungZoona__T3Q-qwen2.5-14b-v1.0-e3-details'

with open(model_name.replace("/", "_") + ".pkl", "wb") as f:
    pickle.dump(models_dict[model_name], f)

In [33]:
with open(model_name.replace("/", "_") + ".pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [34]:
print(single_model_dict.keys())

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])


### Debug

In [20]:
# raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
models_dict = extract_source_outputs_from_v2_raw(raw_path)

100%|██████████| 4/4 [00:00<00:00, 73908.44it/s]


In [21]:
models_dict.keys()

dict_keys(['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details', 'open-llm-leaderboard/LeroyDyer__SpydazWeb_AI_HumanAI_011_INSTRUCT-details', 'open-llm-leaderboard/Youlln__ECE-MIRAGE-1-12B-details', 'open-llm-leaderboard/jaspionjader__bh-39-details'])

In [ ]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']

In [23]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'].keys()

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])

In [24]:
import pickle

with open("matter_0.2-7B-DPO-details.pkl", "wb") as f:
    pickle.dump(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'], f)



In [25]:
with open("matter_0.2-7B-DPO-details.pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [26]:
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["arguments"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["filtered_resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["correctness"]))




12032
12032
12032
12032
